# Voice Note Translator: Kaggle run

Runs the full pipeline (IndicConformer ASR, Indic-to-Indic translation, IndicF5 voice cloning) on a Kaggle GPU, evaluates it on FLEURS, and launches a public demo link.

**Before you run**

1. Settings panel: **Accelerator → GPU T4 x2** (or P100) and **Internet → On** (needs a phone-verified Kaggle account).
2. Accept the terms on https://huggingface.co/ai4bharat/IndicF5 and https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual with the same Hugging Face account as your token.
3. **Add-ons → Secrets**: add `HF_TOKEN` (a read token) and `SARVAM_API_KEY`, and tick both for this notebook.

Then **Run All**. The first run downloads about 5 GB of models and data, so expect 15 to 25 minutes.

In [ ]:
# 1. Get the code
BRANCH = "claude/project-thread-ycbdl8"   # switch to "main" once the PR is merged
!rm -rf RAG-VOICE-SYSTEM && git clone -q -b {BRANCH} https://github.com/AR-droid/RAG-VOICE-SYSTEM.git
%cd RAG-VOICE-SYSTEM
!git log --oneline -1

In [ ]:
# 2. Install (torch and torchaudio are already in the Kaggle image).
# datasets<4 because FLEURS still loads through a dataset script.
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q sarvamai jiwer sacrebleu speechbrain onnxruntime soundfile librosa python-dotenv "datasets<4" "gradio>=4"

In [ ]:
# 3. Secrets and GPU check
import os, torch
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ["HF_TOKEN", "SARVAM_API_KEY"]:
    os.environ[name] = secrets.get_secret(name)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, turn on a GPU accelerator")

In [ ]:
# 4. Load the models once (ASR + translator + TTS)
import time
from voicenote.pipeline import build_default_pipeline

start = time.time()
pipeline = build_default_pipeline()
print(f"Models loaded in {time.time() - start:.0f} s")

## Smoke test

Builds a 5-sentence Tamil→Hindi set from FLEURS and translates one note so you can hear it before the full run.

In [ ]:
!python -m voicenote.eval.make_fleurs_manifest --src ta --tgt hi --n 5 --out data/smoke

In [ ]:
import csv
from IPython.display import Audio, display

row = next(csv.DictReader(open("data/smoke/fleurs_ta-hi.csv", encoding="utf-8")))
result = pipeline.translate_file(os.path.join("data/smoke", row["audio_path"]), "ta", "hi")

print("Tamil transcript :", result.transcript)
print("Hindi translation:", result.translation)
print("Reference (FLEURS):", row["ref_translation"])
print("Timings:", result.timings, f"| RTF {result.real_time_factor:.2f}")
print("Input (Tamil, original speaker):")
display(Audio(os.path.join("data/smoke", row["audio_path"])))
print("Output (Hindi, cloned voice):")
display(Audio(result.audio, rate=result.sample_rate))

## Evaluation

50 notes per language pair. Change `PAIRS` or `N` to taste; each pair takes roughly 10 to 20 minutes on a T4.

In [ ]:
from voicenote.eval.metrics import SpeakerEncoder
from voicenote.eval.run_eval import evaluate, read_manifest, summarize
import pandas as pd

PAIRS = [("ta", "hi"), ("hi", "ta"), ("te", "hi")]
N = 50

encoder = SpeakerEncoder()
all_records = []
for src, tgt in PAIRS:
    !python -m voicenote.eval.make_fleurs_manifest --src {src} --tgt {tgt} --n {N} --out data/eval
    rows = read_manifest(f"data/eval/fleurs_{src}-{tgt}.csv")
    all_records += evaluate(rows, pipeline, encoder, f"results/{src}-{tgt}")

os.makedirs("results", exist_ok=True)
pd.DataFrame(all_records).to_csv("results/results.csv", index=False)
summary = summarize(all_records)
open("results/summary.md", "w").write(summary + "\n")
print(summary)

In [ ]:
# Listen to a few outputs next to their inputs
df = pd.DataFrame(all_records)
for _, r in df.sample(min(3, len(df)), random_state=0).iterrows():
    print(f"{r['pair']}  speaker_sim={r['speaker_sim']}  round_trip_cer={r['round_trip_cer']}")
    print("  ", r["translation"])
    display(Audio(r["audio_path"]))
    display(Audio(os.path.join("results", r["pair"], "audio", r["output"])))

In [ ]:
# Bundle results for download (Output tab → results.zip). Paste summary.md into the README.
!zip -qr /kaggle/working/results.zip results
print(open("results/summary.md").read())

## Live demo

Starts the Gradio app with a public `*.gradio.live` link that lasts up to 72 hours or until the notebook stops. Good for a quick demo; for a permanent link, deploy to a Hugging Face Space.

In [ ]:
import voicenote.app as app

app._pipeline = pipeline   # reuse the loaded models
app.demo.queue().launch(share=True, debug=False)